# Topic 01 - Payoffs, Discounting, Forward Prices, and Put-Call Parity

This is the foundation note for FINM 32000 Numerical Methods for Option Pricing.

The goal is not to memorize formulas first. The goal is to understand what the symbols are saying, then turn that understanding into exam procedures.

Course sources used: Lecture 1, Homework 1, Homework 6 Problem 2, Practice Midterm Problem 1, and the midterm solutions.

## Recommended Study Order

We will go from foundations to final-exam topics in this order:

1. Payoffs, discounting, forward prices, and put-call parity.
2. Black-Scholes formula, implied volatility, and time-dependent volatility.
3. Greeks, Taylor approximation, delta hedging, and dollar gamma.
4. Trees: European, barrier, and American-style options.
5. Finite differences: partial differential equations, explicit, implicit, and Crank-Nicolson.
6. Monte Carlo basics: estimator, variance, standard error, and confidence interval.
7. Variance reduction, excluding importance sampling for the final.
8. Conditional Monte Carlo for Asian-style options.
9. Longstaff-Schwartz for Bermudan and American-style options.
10. Final-exam coding-debug patterns.

Importance sampling is lower priority because the professor said it will not be on the final exam.

## Why This Topic Comes First

Almost every problem in this course starts with a financial contract and turns it into a mathematical object.

If the contract pays something in the future, we need to answer two questions:

1. What exactly is the future payoff?
2. How do we convert that future payoff into a price today?

The first question is about payoff notation. The second question is about discounting and risk-neutral pricing.

If these are unclear, later formulas like Black-Scholes, finite differences, and Monte Carlo will feel like random symbols. They are not random symbols. They are just different numerical methods for computing the same basic object: today's value of a future payoff.

## 1. Basic Notation

The symbol $S_t$ means the price of the underlying asset at time $t$.

Examples:

- $S_0$ is the stock price today.
- $S_T$ is the stock price at expiry time $T$.
- $S_{0.5}$ is the stock price halfway through a one-year period.

The symbol $K$ usually means the strike price.

The symbol $T$ usually means the expiry or payment date.

The notation $x^+$ means the positive part of $x$:

$$x^+ = \max(x,0).$$

So if $x=7$, then $x^+=7$. If $x=-3$, then $x^+=0$.

This notation is used constantly because options often pay only when something is favorable.

## 2. Payoffs

A payoff is the amount of money the contract pays in the future.

A European call option with strike $K$ and expiry $T$ pays:

$$(S_T-K)^+.$$

This means:

- If $S_T>K$, exercise the call and receive $S_T-K$.
- If $S_T\le K$, the call is worthless and pays $0$.

A European put option pays:

$$(K-S_T)^+.$$

This means:

- If $S_T<K$, exercise the put and receive $K-S_T$.
- If $S_T\ge K$, the put is worthless and pays $0$.

A forward contract with delivery price $K$ pays:

$$S_T-K.$$

There is no positive-part symbol here. A forward can have a negative payoff.

## 3. Written Example: Decode Several Payoffs

Suppose $K=100$.

### Call payoff

If $S_T=115$, then:

$$(S_T-K)^+=(115-100)^+=15.$$

If $S_T=90$, then:

$$(S_T-K)^+=(90-100)^+=0.$$

### Put payoff

If $S_T=90$, then:

$$(K-S_T)^+=(100-90)^+=10.$$

If $S_T=115$, then:

$$(K-S_T)^+=(100-115)^+=0.$$

### Forward payoff

If $S_T=115$, then:

$$S_T-K=115-100=15.$$

If $S_T=90$, then:

$$S_T-K=90-100=-10.$$

The forward payoff can be negative. This is why a forward is not the same as a call option.

## 4. Asian-Style Payoff Language

A standard European option depends only on $S_T$, the final stock price.

An Asian-style option depends on an average of stock prices.

For example, the practice final has a payoff at time $T_2=1.0$:

$$\left(\frac{S_{0.5}+S_{1.0}}{2}-12\right)^+.$$

This says:

- Observe the stock price at time $0.5$.
- Observe the stock price at time $1.0$.
- Average those two prices.
- Compare the average to strike $12$.
- Pay the excess if the average is above $12$; otherwise pay $0$.

Example: if $S_{0.5}=11$ and $S_{1.0}=15$, then the average is:

$$\frac{11+15}{2}=13.$$

The payoff is:

$$(13-12)^+=1.$$

Example: if $S_{0.5}=10$ and $S_{1.0}=13$, then the average is:

$$\frac{10+13}{2}=11.5.$$

The payoff is:

$$(11.5-12)^+=0.$$

## 5. Discounting: Why Future Money Is Not Today's Money

If the risk-free interest rate is $r$, then one dollar today grows to $e^{rT}$ dollars at time $T$.

So one dollar paid at time $T$ is worth today:

$$e^{-rT}.$$

More generally, a deterministic payment $A$ at time $T$ is worth today:

$$e^{-rT}A.$$

Example: if $r=5\%$ and $T=2$, then receiving $100$ dollars in two years is worth:

$$e^{-0.05\cdot 2}100.$$

You do not need to simplify this unless the exam asks for a decimal.

## 6. Risk-Neutral Pricing in Plain Language

If a future payoff is random, we cannot just discount one number. We first take the correct pricing average, then discount.

The basic pricing rule is:

$$\text{price today} = E^Q[\text{discounted future payoff}].$$

The symbol $E^Q$ means expectation under the risk-neutral probability measure.

Simple language: risk-neutral probabilities are the artificial probabilities that make discounted tradable asset prices fair, meaning no arbitrage.

They are not necessarily real-world probabilities.

Why do we use them? Because option prices are pinned down by replication and absence of arbitrage, not by investors' real-world opinions. The risk-neutral probability measure is a convenient way to write those no-arbitrage prices as expected discounted payoffs.

## 7. Tiny One-Period Example: Where Risk-Neutral Averaging Comes From

Suppose today $S_0=100$.

At time $1$, the stock will be either:

- $120$ in the up state,
- $80$ in the down state.

Assume the interest rate is zero. We price a call with strike $100$.

The call payoff is:

- Up state: $(120-100)^+=20$.
- Down state: $(80-100)^+=0$.

The risk-neutral probability $p$ is chosen so that the stock price today equals the discounted average future stock price. Since $r=0$, no discounting is needed:

$$100=p\cdot 120+(1-p)\cdot 80.$$

Solving:

$$100=80+40p,$$

$$p=\frac{20}{40}=0.5.$$

Therefore the call price is:

$$0.5\cdot 20+0.5\cdot 0=10.$$

This is the simplest version of risk-neutral pricing. Later, trees, Black-Scholes, partial differential equations, and Monte Carlo are all more advanced ways to compute the same kind of no-arbitrage average.

## 8. The Discounted Payoff Variable $Y$

In Monte Carlo notes, the professor often writes:

$$Y = \text{discounted payoff}.$$

For a European call, the future payoff is:

$$(S_T-K)^+.$$

So the discounted payoff is:

$$Y=e^{-rT}(S_T-K)^+.$$

For the Asian-style call from the practice final, the future payoff at time $T_2$ is:

$$\left(\frac{S_{T_1}+S_{T_2}}{2}-K\right)^+.$$

So the discounted payoff is:

$$Y=e^{-rT_2}\left(\frac{S_{T_1}+S_{T_2}}{2}-K\right)^+.$$

This is not a mysterious new formula. It is just:

$$Y = \text{today's value of the random future payoff on one simulated path}.$$

Then the price is:

$$C_0=E^Q[Y].$$

## 9. Forward Contract Versus Forward Price

These two phrases sound similar but mean different things.

A forward contract with delivery price $K$ pays:

$$S_T-K.$$

The forward price $F_0$ is the special delivery price that makes the forward contract worth zero today.

So:

$$\text{value today of payoff } S_T-F_0 = 0.$$

If the underlying pays no dividends, then:

$$F_0=S_0e^{rT}.$$

If the underlying pays continuous dividend yield $q$, then:

$$F_0=S_0e^{(r-q)T}.$$

The dividend yield lowers the forward price because owning the stock gives dividend benefits before expiry. A forward buyer does not receive those dividends before delivery.

## 10. Time-$t$ Forward Price

At time $t$, for delivery at time $T$, the forward price is:

$$F_t=S_te^{(r-q)(T-t)}.$$

Special cases:

- If there are no dividends, $q=0$, so $F_t=S_te^{r(T-t)}$.
- If interest rates are zero and dividends are zero, then $F_t=S_t$.

The value at time $t$ of a forward contract with delivery price $K$ is:

$$e^{-r(T-t)}(F_t-K).$$

This formula says: the forward contract pays $S_T-K$, whose fair forward value is $F_t-K$, then we discount that future amount back to time $t$.

## 11. Written Example: Forward Price With Dividends

Suppose:

- $S_0=100$,
- $r=5\%$,
- $q=2\%$,
- $T=1$ year.

Then the one-year forward price is:

$$F_0=100e^{(0.05-0.02)\cdot 1}=100e^{0.03}.$$

If a forward contract instead has delivery price $K=100$, its time-0 value is:

$$e^{-0.05}(100e^{0.03}-100).$$

This value is positive because the agreed delivery price $100$ is below the fair forward price $100e^{0.03}$.

## 12. Put-Call Parity: The Most Important Identity In This Topic

A call minus a put with the same strike and expiry has payoff:

$$(S_T-K)^+-(K-S_T)^+.$$

Now check the two cases.

If $S_T>K$:

- Call pays $S_T-K$.
- Put pays $0$.
- Difference is $S_T-K$.

If $S_T<K$:

- Call pays $0$.
- Put pays $K-S_T$.
- Difference is $-(K-S_T)=S_T-K$.

So in every state:

$$(S_T-K)^+-(K-S_T)^+=S_T-K.$$

That means:

$$\text{call} - \text{put} = \text{forward contract}.$$

Therefore:

$$C_0-P_0=e^{-rT}(F_0-K).$$

This is put-call parity written using the forward price $F_0$.

## 13. How To Use Put-Call Parity On Exams

Start from:

$$C_0-P_0=e^{-rT}(F_0-K).$$

If solving for forward price:

$$F_0=K+e^{rT}(C_0-P_0).$$

If solving for a missing put price:

$$P_0=C_0-e^{-rT}(F_0-K).$$

If solving for a missing call price:

$$C_0=P_0+e^{-rT}(F_0-K).$$

Exam habit: before plugging in numbers, ask which thing is unknown: $F_0$, $C_0$, or $P_0$.

## 14. Practice Midterm Example: Infer The Forward Price

Practice Midterm Problem 1 gives:

- Strike $K=2280$,
- Call price $C_0=44.7$,
- Put price $P_0=59.2$,
- Interest rate $r=0.0432$,
- Expiry $T=25/365$.

Use put-call parity:

$$C_0-P_0=e^{-rT}(F_0-K).$$

Solve for $F_0$:

$$F_0=K+e^{rT}(C_0-P_0).$$

Plug in:

$$F_0=2280+e^{0.0432\cdot 25/365}(44.7-59.2).$$

Because the put is more expensive than the call at the same strike, $C_0-P_0$ is negative. So the inferred forward price is below $2280$.

That sign check is important. It helps catch algebra mistakes.

## 15. Practice Midterm Example: Value Of A Forward Contract

Now suppose we want the value of a forward contract with delivery price $2200$ and the same expiry.

The formula is:

$$\text{forward contract value}=e^{-rT}(F_0-K).$$

Here the delivery price is $K=2200$, not $2280$.

$$\text{value}=e^{-0.0432\cdot 25/365}(F_0-2200).$$

Using the previous expression for $F_0$:

$$F_0=2280+e^{rT}(44.7-59.2).$$

So:

$$e^{-rT}(F_0-2200)=80e^{-rT}+44.7-59.2.$$

This is the form shown in the practice midterm solution.

Why does it simplify that way? Because:

$$F_0-2200=(2280-2200)+e^{rT}(44.7-59.2).$$

Then multiplying by $e^{-rT}$ gives:

$$80e^{-rT}+44.7-59.2.$$

## 16. Practice Midterm Example: Fill A Missing Put Price

The practice midterm gives a call price at strike $2300$:

$$C_0(2300)=35.4.$$

We want the missing put price $P_0(2300)$.

Use put-call parity at the same strike:

$$C_0(2300)-P_0(2300)=e^{-rT}(F_0-2300).$$

Solve for the put:

$$P_0(2300)=C_0(2300)-e^{-rT}(F_0-2300).$$

So:

$$P_0(2300)=35.4-e^{-0.0432\cdot 25/365}(F_0-2300).$$

Using the earlier relationship, this can also be written as:

$$35.4+20e^{-0.0432\cdot 25/365}-44.7+59.2.$$

Exam warning: put-call parity relates calls and puts with the same strike. Do not accidentally use a different strike unless you have already inferred the forward price and are applying parity again.

## 17. Common Exam Mistakes

Mistake 1: Confusing forward price with forward contract value.

- Forward price: the delivery price that makes the contract worth zero.
- Forward contract value: today's value of a contract with a specified delivery price.

Mistake 2: Forgetting the discount factor in put-call parity.

Correct:

$$C_0-P_0=e^{-rT}(F_0-K).$$

Mistake 3: Using real-world probabilities instead of risk-neutral probabilities.

Option pricing is based on no-arbitrage pricing, so we use risk-neutral probabilities.

Mistake 4: Treating a positive-part payoff like an ordinary expression.

$$(S_T-K)^+$$

is not the same as $S_T-K$. The positive-part symbol floors the payoff at zero.

Mistake 5: Applying put-call parity to different strikes.

The direct parity relationship uses the same strike and same expiry.

## 18. Exam Procedure Sheet

When you see a pricing problem, do this:

1. Identify the payoff.
2. Identify the payment time.
3. Discount from payment time to today.
4. If probabilities are needed, use risk-neutral probabilities.
5. If calls and puts with same strike and expiry appear, try put-call parity.
6. If a forward price appears, remember it is not the same as the value of a forward contract.

Put-call parity formulas to memorize:

$$C_0-P_0=e^{-rT}(F_0-K),$$

$$F_0=K+e^{rT}(C_0-P_0),$$

$$P_0=C_0-e^{-rT}(F_0-K).$$

## 19. Exercises

### Exercise 1

A stock has $S_T=120$ at expiry. Compute the payoff of:

1. A call with $K=100$.
2. A put with $K=100$.
3. A forward with delivery price $K=100$.

### Exercise 2

A call has price $8$, a put has price $5$, the strike is $100$, $r=0.04$, and $T=0.5$. Find the forward price $F_0$.

### Exercise 3

Suppose $F_0=105$, $r=0.03$, $T=1$, and a call with strike $100$ has price $12$. Use put-call parity to find the put price.

### Exercise 4

An Asian-style call pays:

$$\left(\frac{S_{0.25}+S_{0.5}+S_{1.0}}{3}-50\right)^+.$$

If $S_{0.25}=45$, $S_{0.5}=54$, and $S_{1.0}=60$, what is the payoff?

### Exercise 5

A dividend-paying stock has $S_0=200$, $r=6\%$, $q=2\%$, and $T=2$. Find the forward price.

## 20. Exercise Answers

### Answer 1

Call:

$$(120-100)^+=20.$$

Put:

$$(100-120)^+=0.$$

Forward:

$$120-100=20.$$

### Answer 2

Use:

$$F_0=K+e^{rT}(C_0-P_0).$$

So:

$$F_0=100+e^{0.04\cdot 0.5}(8-5).$$

$$F_0=100+3e^{0.02}.$$

### Answer 3

Use:

$$P_0=C_0-e^{-rT}(F_0-K).$$

So:

$$P_0=12-e^{-0.03}(105-100).$$

$$P_0=12-5e^{-0.03}.$$

### Answer 4

Average price:

$$\frac{45+54+60}{3}=53.$$

Payoff:

$$(53-50)^+=3.$$

### Answer 5

Use:

$$F_0=S_0e^{(r-q)T}.$$

So:

$$F_0=200e^{(0.06-0.02)\cdot 2}=200e^{0.08}.$$

## 21. What You Should Be Able To Do After This Topic

You should be able to:

1. Read payoff notation without panic.
2. Explain why option prices are expected discounted payoffs under risk-neutral probabilities.
3. Distinguish payoff, discounted payoff, and price.
4. Distinguish forward price from forward contract value.
5. Use put-call parity to infer a forward price or a missing option price.
6. Catch common sign and discounting mistakes.

These are the base tools for Black-Scholes, trees, finite differences, and Monte Carlo.